# Solar PV Output Sanity Check (Issue #6)

This notebook verifies the 15-minute simulated solar generation data produced by `pvlib` for the 600 kWp rooftop installation in Ashulia, Dhaka.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from energy_brain.config import load_site_config

cfg = load_site_config()
pv_df = pd.read_parquet("../data/processed/pv.parquet")
pv_df["timestamp"] = pd.to_datetime(pv_df["timestamp"])
pv_df.set_index("timestamp", inplace=True)

print(f"Total slots: {len(pv_df)}")
print(f"Peak output: {pv_df['pv_kw'].max():.2f} kW (System rating: {cfg.solar.capacity_kwp} kWp)")
pv_df.head()

## 1. Single Day Check: Clear Day Profile
Verifying zero generation before dawn and after dusk, with peak near solar noon (12:00 - 13:00).

In [ ]:
sample_day = pv_df.loc["2024-03-15"]
plt.figure(figsize=(10, 4))
plt.plot(sample_day.index.strftime("%H:%M"), sample_day["pv_kw"], color="goldenrod", lw=2)
plt.xticks(sample_day.index.strftime("%H:%M")[::8], rotation=45)
plt.title("15-Minute Solar PV AC Output (Dhaka - 2024-03-15)")
plt.xlabel("Time of Day (Asia/Dhaka)")
plt.ylabel("AC Power (kW)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Weekly Profile (Summer / Monsoon)
Exhibiting day-to-day cloud intermittency.

In [ ]:
sample_week = pv_df.loc["2024-06-01":"2024-06-07"]
plt.figure(figsize=(12, 4))
plt.plot(sample_week.index, sample_week["pv_kw"], color="tab:orange", lw=1.5)
plt.title("Weekly Solar PV Generation (Dhaka - June 2024 Monsoon Intermittency)")
plt.xlabel("Date")
plt.ylabel("AC Power (kW)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Annual Generation & Specific Yield
Sanity check against Bangladesh benchmark (~1100 - 1300 kWh/kWp/year).

In [ ]:
annual_energy_mwh = pv_df["pv_kw"].groupby(pv_df.index.year).sum() * 0.25 / 1000.0
specific_yield = annual_energy_mwh * 1000.0 / cfg.solar.capacity_kwp
metrics_df = pd.DataFrame(
    {
        "Annual Generation (MWh)": annual_energy_mwh,
        "Specific Yield (kWh/kWp/yr)": specific_yield,
    }
)
metrics_df